In [1]:
import json

with open("notion_data.json", "r", encoding="utf-8") as f:
    documents = json.load(f)

print("Pages:", len(documents))

Pages: 45


In [2]:
def create_sections(document):

    sections = []

    current_section = "Introduction"
    current_blocks = []

    for block in document["blocks"]:

        text = block["text"].strip()

        if not text:
            continue

        # Heading = new section
        if block["type"].startswith("heading_"):

            # Save previous section
            if current_blocks:
                sections.append({
                    "page_id": document["page_id"],
                    "page_title": document["title"],
                    "page_url": document["url"],
                    "section": current_section,
                    "blocks": current_blocks
                })

            current_section = text
            current_blocks = []

        else:
            current_blocks.append(block)

    # Save last section
    if current_blocks:
        sections.append({
            "page_id": document["page_id"],
            "page_title": document["title"],
            "page_url": document["url"],
            "section": current_section,
            "blocks": current_blocks
        })

    return sections

In [3]:
all_sections = []

for document in documents:

    sections = create_sections(document)

    all_sections.extend(sections)

print("Total sections:", len(all_sections))

Total sections: 3151


In [4]:
for section in all_sections:

    text = "\n".join(
        block["text"]
        for block in section["blocks"]
    )

    section["text"] = text
    section["char_count"] = len(text)
    section["word_count"] = len(text.split())

In [6]:
for section in all_sections:

    print(
        f"{section['page_title']} | "
        f"{section['section']} | "
        f"{section['word_count']} words | "
        f"{section['char_count']} chars"
    )

Agent Evaluation | Introduction | 8 words | 49 chars
Agent Evaluation | 1. LLM-as-a-Judge | 79 words | 510 chars
Agent Evaluation | 2. Human Annotation | 40 words | 275 chars
Agent Evaluation | 3. Code-Based Evals | 84 words | 600 chars
Agent Evaluation | The key difference | 101 words | 602 chars
AI Roadmap | Prompt Engineering | 65 words | 438 chars
AI Roadmap | Fine Tuning: | 38 words | 265 chars
AI Roadmap | Rag | 36 words | 256 chars
AI Roadmap | AI Agent: | 6 words | 39 chars
AI Roadmap | Responsible AI principles | 67 words | 746 chars
AI Roadmap | What is LangChain? | 44 words | 332 chars
AI Roadmap | What are Document Loaders? | 40 words | 278 chars
AI Roadmap | Document Object Structure | 4 words | 30 chars
AI Roadmap | Creating a Document Object | 24 words | 250 chars
AI Roadmap | 1. File-Based Loaders | 12 words | 66 chars
AI Roadmap | 2. Source-Based Loaders | 11 words | 79 chars
AI Roadmap | CSV Loader | 8 words | 47 chars
AI Roadmap | Example | 10 words | 191 chars
AI Ro

In [7]:
largest_sections = sorted(
    all_sections,
    key=lambda x: x["word_count"],
    reverse=True
)

for section in largest_sections[:20]:

    print(
        f"{section['page_title']} | "
        f"{section['section']} | "
        f"{section['word_count']} words"
    )

AI Roadmap | Simple Mental Model | 1246 words
OOPS | Introduction | 1164 words
Design Patterns - AI Agent | Introduction | 1091 words
AI Roadmap | ⚠️ The Bottleneck Problem | 809 words
NLP | ⚠️ The Bottleneck Problem | 809 words
ARM Work I do | Built a pipeline to push the Vite app to NetStorage | 664 words
OOPS | Class vs. Instance Variable | 584 words
ARM Work I do | KQL + Power BI | 439 words
TAX 2025-26 | Introduction | 428 words
AI Roadmap | The Problem It Solves | 352 words
NLP | The Problem It Solves | 352 words
AI Roadmap | The Problem It Solves | 321 words
NLP | The Problem It Solves | 321 words
OOPS | Solution 2: super() and the Diamond Problem | 317 words
Rag Theory | Key Embedding Concepts | 306 words
100 days of AI | One Neuron Model (Single Input) | 306 words
AI | One Neuron Model (Single Input) | 306 words
python | What does await do? | 296 words
React JS | React Router: | 286 words
flask | Introduction | 283 words


In [9]:
import json

MAX_WORDS = 200
OVERLAP_WORDS = 20


def split_text(text, max_words=MAX_WORDS, overlap=OVERLAP_WORDS):
    """
    Split text into chunks of at most max_words.
    Uses overlap between consecutive chunks.
    """

    words = text.split()

    # No splitting needed
    if len(words) <= max_words:
        return [text]

    chunks = []

    start = 0

    while start < len(words):

        end = min(start + max_words, len(words))

        chunk = " ".join(words[start:end])
        chunks.append(chunk)

        # Reached the end
        if end >= len(words):
            break

        # Start next chunk with overlap
        start = end - overlap

    return chunks


def create_sections(document):
    """
    Group Notion blocks into sections.

    A new heading starts a new section.
    """

    sections = []

    current_section = "Introduction"
    current_blocks = []

    for block in document["blocks"]:

        text = block["text"].strip()

        if not text:
            continue

        # Heading → start a new section
        if block["type"].startswith("heading_"):

            # Save previous section
            if current_blocks:

                sections.append({
                    "page_id": document["page_id"],
                    "page_title": document["title"],
                    "page_url": document["url"],
                    "section": current_section,
                    "blocks": current_blocks
                })

            current_section = text
            current_blocks = []

        else:
            current_blocks.append(block)

    # Save final section
    if current_blocks:

        sections.append({
            "page_id": document["page_id"],
            "page_title": document["title"],
            "page_url": document["url"],
            "section": current_section,
            "blocks": current_blocks
        })

    return sections


# ============================================================
# 1. LOAD NOTION DATA
# ============================================================

with open("notion_data.json", "r", encoding="utf-8") as f:
    documents = json.load(f)


# ============================================================
# 2. CREATE CHUNKS
# ============================================================

chunks = []

for document in documents:

    sections = create_sections(document)

    for section in sections:

        # ----------------------------------------------------
        # Merge all blocks in the section
        # ----------------------------------------------------

        merged_text = "\n".join(
            block["text"]
            for block in section["blocks"]
        ).strip()

        if not merged_text:
            continue

        # ----------------------------------------------------
        # Preserve original Notion block IDs
        # ----------------------------------------------------

        source_block_ids = [
            block["block_id"]
            for block in section["blocks"]
        ]

        # ----------------------------------------------------
        # Split section if > MAX_WORDS
        # ----------------------------------------------------

        section_chunks = split_text(merged_text)

        for chunk_text in section_chunks:

            chunks.append({
                "chunk_id": None,

                "text": chunk_text,

                "previous_chunk_id": None,
                "next_chunk_id": None,

                "metadata": {
                    "page_id": section["page_id"],
                    "page_title": section["page_title"],
                    "page_url": section["page_url"],
                    "section": section["section"],
                    "source_block_ids": source_block_ids
                }
            })


# ============================================================
# 3. ASSIGN CHUNK IDs
# ============================================================

for index, chunk in enumerate(chunks):

    page_id = chunk["metadata"]["page_id"]

    chunk["chunk_id"] = f"{page_id}_{index + 1:04d}"


# ============================================================
# 4. LINK PREVIOUS / NEXT CHUNKS
# ============================================================

for index, chunk in enumerate(chunks):

    # Previous chunk
    if index > 0:
        chunk["previous_chunk_id"] = chunks[index - 1]["chunk_id"]

    # Next chunk
    if index < len(chunks) - 1:
        chunk["next_chunk_id"] = chunks[index + 1]["chunk_id"]


# ============================================================
# 5. SAVE
# ============================================================

with open("chunks.json", "w", encoding="utf-8") as f:

    json.dump(
        chunks,
        f,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# 6. BASIC VALIDATION
# ============================================================

print("Documents:", len(documents))
print("Total chunks:", len(chunks))

max_words = max(
    len(chunk["text"].split())
    for chunk in chunks
)

print("Maximum words in a chunk:", max_words)

print("\nFirst chunk:")
print(json.dumps(chunks[0], indent=2, ensure_ascii=False))

print("\nLast chunk:")
print(json.dumps(chunks[-1], indent=2, ensure_ascii=False))

print("\nSaved → chunks.json")

Documents: 45
Total chunks: 3225
Maximum words in a chunk: 200

First chunk:
{
  "chunk_id": "3d5fca33-84fc-80fe-aac2-d743eaa88e60_0001",
  "text": "LLM as a judge \nHuman Annotation\nCode-based Evals",
  "previous_chunk_id": null,
  "next_chunk_id": "3d5fca33-84fc-80fe-aac2-d743eaa88e60_0002",
  "metadata": {
    "page_id": "3d5fca33-84fc-80fe-aac2-d743eaa88e60",
    "page_title": "Agent Evaluation",
    "page_url": "https://app.notion.com/p/Agent-Evaluation-3d5fca3384fc80feaac2d743eaa88e60",
    "section": "Introduction",
    "source_block_ids": [
      "3d5fca33-84fc-807b-a065-d261ab157259",
      "3d5fca33-84fc-8080-98ef-d88022503627",
      "3d5fca33-84fc-801e-b9f1-c439c11d1caf"
    ]
  }
}

Last chunk:
{
  "chunk_id": "204fca33-84fc-801f-b7d1-d4fc714f41f9_3225",
  "text": "Creating an env: Python-m venv env \nActivate the virtual environment: env\\Scripts\\activate\nGet requirement.txt: pip freeze > requirements.txt",
  "previous_chunk_id": "22bfca33-84fc-80d8-90e7-d462a279c782_3